In [1]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': True,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8],
  'custom1': [9, 11]},
 'method': 'welch',
 'stepSize': 0.3,
 'windowLength': 3}

In [3]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [5]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [6]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx16g -Xms8g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # SPARK DIRECTORY FOR THREADS / PERSIST
    
    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \
    
    # THIS IS WHERE SPARK WILL PUT ITS TEMPERARY VARIABLES 
    # .config("spark.local.dir", os.path.expanduser("~/external-spark-tmp"))

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "512") \

    # prevent breaking pipes 
    .config("spark.reducer.maxReqsInFlight", "1") \
    .config("spark.shuffle.io.preferDirectBufs", "false") \
    .config("spark.shuffle.file.buffer", "32k") \
    
    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx16g -Xms8g
Picked up _JAVA_OPTIONS: -Xmx16g -Xms8g
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/24 08:56:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


New Spark session created successfully


25/04/24 08:56:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [7]:
for item in spark.sparkContext.getConf().getAll():
    print(item)


('spark.app.submitTime', '1745499416325')
('spark.driver.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false')
('spark.default.parallelism', '12')
('spark.driver.host', '127.0.

In [8]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction_helper.py"))
    print("Added feature_extraction_helper.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added feature_extraction_helper.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


In [10]:
%%time
alz_df_spark = spark.read.parquet("features_alz_extra_features_Apr24_0821.parquet")
cntrl_df_spark = spark.read.parquet("features_cntrl_extra_features_Apr24_0821.parquet")

CPU times: user 1.67 ms, sys: 2.36 ms, total: 4.04 ms
Wall time: 1.13 s


In [11]:
#just renaming things now that we understand the types and where things are coming from
alz_df = alz_df_spark
cntrl_df = cntrl_df_spark

In [12]:
alz_df.show()

+---------+-------+---------+--------+----------------+-------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|     FeatureName| FeatureValue|table_type|
+---------+-------+---------+--------+----------------+-------------+----------+
|  sub-010|   ep-0|      Fp1|    NULL|      TotalPower|  0.011235955| electrode|
|  sub-010|   ep-0|      Fp1|    NULL|     TotalEnergy|   0.21298556| electrode|
|  sub-010|   ep-0|      Fp1|    NULL| SpectralEntropy|    3.1180124| electrode|
|  sub-010|   ep-0|      Fp1|    NULL|  HjorthActivity|4.3815498E-10| electrode|
|  sub-010|   ep-0|      Fp1|    NULL|  HjorthMobility|  0.053108174| electrode|
|  sub-010|   ep-0|      Fp1|    NULL|HjorthComplexity|     6.092837| electrode|
|  sub-010|   ep-0|      Fp1|    NULL|     HjorthIndex|     953.6603| electrode|
|  sub-010|   ep-0|      Fp1|   Alpha|           Power|  0.001618094|      band|
|  sub-010|   ep-0|      Fp1|   Alpha| SpectralEntropy|    3.1425395|      band|
|  sub-010|   ep-0|      Fp1

# Raw Data Visualization

# Start of data processing

In [13]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(8).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(8).persist()

In [14]:
# union everything
full_df = alz_df.unionByName(cntrl_df)

In [15]:
# Split based on feature type
from pyspark.sql.functions import col

band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")


In [16]:
alz_df.unpersist()
cntrl_df.unpersist()

DataFrame[SubjectID: string, EpochID: string, Electrode: string, WaveBand: string, FeatureName: string, FeatureValue: float, table_type: string, label: int]

In [17]:
from pyspark.sql.functions import col

# Filter and save each to Parquet
full_df.filter(col("table_type") == "band") \
    .write.mode("overwrite").parquet("tempParquets/band_df")

full_df.filter(col("table_type") == "electrode") \
    .write.mode("overwrite").parquet("tempParquets/channel_df")

full_df.filter(col("table_type") == "epoch") \
    .write.mode("overwrite").parquet("tempParquets/epoch_df")

In [18]:
import os
os.system('say "START!"')

0

In [18]:
from pyspark.sql.functions import first
from pyspark.sql.functions import concat_ws


# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName")).repartition(100, "SubjectID").persist()

# band_df.write.mode("overwrite").parquet("band_pre_pivot")
# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))
band_df.unpersist()

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName")).repartition(100, "SubjectID").persist()
# channel_df.write.mode("overwrite").parquet("channel_df_pre_pivot")

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))
channel_pivot.unpersist()


# Band-level: Electrode_WaveBand_Feature
epoch_df = epoch_df.withColumn("pivot", col("FeatureName")).repartition(100, "SubjectID").persist()
# epoch_df.write.mode("overwrite").parquet("epoch_df_pre_pivot")


# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))
epoch_pivot.unpersist()

DataFrame[SubjectID: string, EpochID: string, label: int, HjorthMobility: float, Mean: float, RMS: float, Std: float, Variance: float]

In [20]:
import os
os.system('say "DONE!"')

0

In [19]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

In [20]:
from functools import reduce

# full_df = reduce(
#     lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
#     [band_pivot, channel_pivot, epoch_pivot]
# ).fillna(0.0)


full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [band_pivot, channel_pivot, epoch_pivot]# band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)
full_df.repartition(16).persist()


25/04/24 09:03:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_HjorthComplexity: float, C3_Alpha_HjorthIndex: float, C3_Alpha_HjorthMobility: float, C3_Alpha_Power: float, C3_Alpha_SpectralEntropy: float, C3_Beta_HjorthComplexity: float, C3_Beta_HjorthIndex: float, C3_Beta_HjorthMobility: float, C3_Beta_Power: float, C3_Beta_SpectralEntropy: float, C3_Delta_HjorthComplexity: float, C3_Delta_HjorthIndex: float, C3_Delta_HjorthMobility: float, C3_Delta_Power: float, C3_Delta_SpectralEntropy: float, C3_Theta_HjorthComplexity: float, C3_Theta_HjorthIndex: float, C3_Theta_HjorthMobility: float, C3_Theta_Power: float, C3_Theta_SpectralEntropy: float, C3_custom1_HjorthComplexity: float, C3_custom1_HjorthIndex: float, C3_custom1_HjorthMobility: float, C3_custom1_Power: float, C3_custom1_SpectralEntropy: float, C4_Alpha_HjorthComplexity: float, C4_Alpha_HjorthIndex: float, C4_Alpha_HjorthMobility: float, C4_Alpha_Power: float, C4_Alpha_SpectralEntropy: float, C4_Beta_HjorthComplexity: float

In [27]:
band_pivot.unpersist()
channel_pivot.unpersist()
epoch_pivot.unpersist()

DataFrame[SubjectID: string, EpochID: string, label: int, HjorthMobility: float, Mean: float, RMS: float, Std: float, Variance: float]

In [28]:
type(full_df)

pyspark.sql.dataframe.DataFrame

In [29]:
full_df.repartition(16).persist()


25/04/24 09:05:08 WARN CacheManager: Asked to cache already cached data.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_HjorthComplexity: float, C3_Alpha_HjorthIndex: float, C3_Alpha_HjorthMobility: float, C3_Alpha_Power: float, C3_Alpha_SpectralEntropy: float, C3_Beta_HjorthComplexity: float, C3_Beta_HjorthIndex: float, C3_Beta_HjorthMobility: float, C3_Beta_Power: float, C3_Beta_SpectralEntropy: float, C3_Delta_HjorthComplexity: float, C3_Delta_HjorthIndex: float, C3_Delta_HjorthMobility: float, C3_Delta_Power: float, C3_Delta_SpectralEntropy: float, C3_Theta_HjorthComplexity: float, C3_Theta_HjorthIndex: float, C3_Theta_HjorthMobility: float, C3_Theta_Power: float, C3_Theta_SpectralEntropy: float, C3_custom1_HjorthComplexity: float, C3_custom1_HjorthIndex: float, C3_custom1_HjorthMobility: float, C3_custom1_Power: float, C3_custom1_SpectralEntropy: float, C4_Alpha_HjorthComplexity: float, C4_Alpha_HjorthIndex: float, C4_Alpha_HjorthMobility: float, C4_Alpha_Power: float, C4_Alpha_SpectralEntropy: float, C4_Beta_HjorthComplexity: float

In [30]:
full_df.write.mode("overwrite").parquet("tempParquets/full_df_UNFILTERED")

25/04/24 09:05:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 09:05:55 WARN DAGScheduler: Broadcasting large task binary with size 1233.8 KiB
                                                                                

In [31]:
print("w")

w


In [32]:
NUM_TEST_SUBJECTS_PER_GROUP = 2

# Get test subject IDs from full_df (which has .label)
alz_test_subjects = (
    full_df.filter("label == 1")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

cntrl_test_subjects = (
    full_df.filter("label == 0")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

test_subjects = alz_test_subjects + cntrl_test_subjects #this will be the firs 2 subjects of each group for reproduceablility

In [33]:
print("w")

w


In [30]:
# from pyspark.sql.functions import rand

# NUM_TEST_SUBJECTS_PER_GROUP = 2
# SEED = 42

# # Alzheimer's test subjects (label == 1)
# alz_test_subjects = (
#     full_df.filter("label == 1")
#     .select("SubjectID")
#     .distinct()
#     .orderBy(rand(SEED))  # Randomize with seed
#     .limit(NUM_TEST_SUBJECTS_PER_GROUP)
#     .rdd.flatMap(lambda row: row)
#     .collect()
# )

# # Control test subjects (label == 0)
# cntrl_test_subjects = (
#     full_df.filter("label == 0")
#     .select("SubjectID")
#     .distinct()
#     .orderBy(rand(SEED + 1))  # Different seed for different shuffle
#     .limit(NUM_TEST_SUBJECTS_PER_GROUP)
#     .rdd.flatMap(lambda row: row)
#     .collect()
# )

# # Combine test subjects
# test_subjects = alz_test_subjects + cntrl_test_subjects


In [34]:
print(f"alz_test_subjects {alz_test_subjects}")
print(f"cntrl_test_subjects {cntrl_test_subjects}")

alz_test_subjects ['sub-001', 'sub-002']
cntrl_test_subjects ['sub-037', 'sub-038']


In [238]:
# Split into test and train sets
train_df = full_df.filter(~col("SubjectID").isin(test_subjects))
test_df = full_df.filter(col("SubjectID").isin(test_subjects))


In [239]:
print("got here") 

got here


# DO T-TEST HERE !!

In [240]:
len(train_df.columns)

616

# you are here , figure it out !

In [35]:
# import dimensionality_reduction
# import importlib
# importlib.reload(dimensionality_reduction)
# from dimensionality_reduction import min_max_normalize, normalize_by_column_per_subject_wide
# feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]


# # train_norm_df, test_norm_df = min_max_normalize(train_df, test_df, feature_cols)
# train_norm_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_norm_df = normalize_by_column_per_subject_wide(test_df, feature_cols)

In [109]:
train_df.head(1)

25/04/24 10:57:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

[Row(SubjectID='sub-003', EpochID='ep-103', label=1, C3_Alpha_HjorthComplexity=6.130146503448486, C3_Alpha_HjorthIndex=1083.575927734375, C3_Alpha_HjorthMobility=0.046671587973833084, C3_Alpha_Power=0.0021793830674141645, C3_Alpha_SpectralEntropy=2.9399330615997314, C3_Beta_HjorthComplexity=6.130146503448486, C3_Beta_HjorthIndex=1083.575927734375, C3_Beta_HjorthMobility=0.046671587973833084, C3_Beta_Power=0.0007467991090379655, C3_Beta_SpectralEntropy=4.675412654876709, C3_Delta_HjorthComplexity=6.130146503448486, C3_Delta_HjorthIndex=1083.575927734375, C3_Delta_HjorthMobility=0.046671587973833084, C3_Delta_Power=0.07590913772583008, C3_Delta_SpectralEntropy=2.017423629760742, C3_Theta_HjorthComplexity=6.130146503448486, C3_Theta_HjorthIndex=1083.575927734375, C3_Theta_HjorthMobility=0.046671587973833084, C3_Theta_Power=0.008209975436329842, C3_Theta_SpectralEntropy=3.200477361679077, C3_custom1_HjorthComplexity=6.130146503448486, C3_custom1_HjorthIndex=1083.575927734375, C3_custom1_Hj

# NORMALIZING SUJBECT WIDE FOR EXPERIMENT< REMOVE!, MIN MAX< TRY DIFFERNET COMBOS FOR LOWER COHEN!

In [37]:
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide

# normalize_by_column_per_subject_wide(

In [38]:
# train_df.schema

In [39]:
results = []

In [40]:
# import dimensionality_reduction
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide
feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]

# train_norm_df, test_norm_df = train_df, test_df

# train_norm_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_norm_df = normalize_by_column_per_subject_wide(test_df, feature_cols)

# RUNNING T-TEST - using regression to do the test like named here 
# https://stackoverflow.com/questions/58851008/how-to-perform-student-t-test-in-pyspark

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

results = []
for feat in feature_cols:
    # Step 1: Assemble the single feature into featuresCol
    assembler = VectorAssembler(inputCols=["label"], outputCol="features")
    assembled = assembler.transform(train_df.select("label", feat).dropna())

    # Step 2: Fit regression model: feature ~ label
    lr = LinearRegression(featuresCol="features", labelCol=feat, regParam=0)
    model = lr.fit(assembled)

    # Step 3: Get t-stat and p-value for 'label'
    summary = model.summary
    t_stat = summary.tValues[1]  # index 1 corresponds to label coefficient
    p_val = summary.pValues[1]

    # Save results
    results.append((feat, t_stat, p_val))

    # drop the model, no need for it to stay in memorry
    assembled.unpersist(blocking=True)
    import gc
    gc.collect()
    from pyspark import SparkContext
    SparkContext._jvm.java.lang.System.gc()






25/04/24 09:07:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 09:08:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 09:08:48 WARN Instrumentation: [f96f03ac] regParam is zero, which might cause numerical instability and overfitting.
25/04/24 09:08:49 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/04/24 09:08:49 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
25/04/24 09:09:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 09:10:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 09:10:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 09:10:57 WARN Instrumentation: [e403ff98] regParam is zero, wh

KeyboardInterrupt: 

In [55]:
results_df = pd.DataFrame(results, columns=["Feature", "T_statistic", "P_value"])
results_df.sort_values("P_value", inplace=True)  # sort by significance


# add benferroni or FDR correction 
# from statsmodels.stats.multitest import multipletests

# # Apply FDR correction
# rejected, pvals_corrected, _, _ = multipletests(results_df["P_value"], alpha=0.05, method="fdr_bh")
# results_df["FDR_corrected"] = pvals_corrected
# results_df["Significant"] = rejected


# train_norm_df.repartition(16).persist()
# test_norm_df.repartition(16).persist()
print("finished T-test and have results")

finished T-test and have results


In [ ]:
os.system('say "t-test is done!"')

In [41]:
# train_norm_df.head(1)
results_df


NameError: name 'results_df' is not defined

25/04/24 09:50:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

In [ ]:
results_df = results_df.sort_values(by="T_statistic", ascending=False)

In [42]:
results_df = []

In [43]:
# results_df.to_pickle("ttest_results.pkl")


In [110]:
import gc
gc.collect()
from pyspark import SparkContext
SparkContext._jvm.java.lang.System.gc()

In [44]:
from pyspark.sql.functions import col, avg, stddev, count

cohen_d_results = []

# remember 1 is alzeimers and 0 is control, so if negative cohen, means control's mean was larger 
for feat in feature_cols:
    stats = (
        train_df
        .select("label", feat)
        .dropna()
        .groupBy("label")
        .agg(
            avg(feat).alias("mean"),
            stddev(feat).alias("std"),
            count(feat).alias("n")
        )
        .toPandas()
        .set_index("label")
    )
    
    if 0 in stats.index and 1 in stats.index:
        mean0 = stats.loc[0, "mean"]
        mean1 = stats.loc[1, "mean"]
        std0 = stats.loc[0, "std"]
        std1 = stats.loc[1, "std"]
        n0 = stats.loc[0, "n"]
        n1 = stats.loc[1, "n"]

        # pooled std
        pooled_std = (( (n0 - 1) * std0**2 + (n1 - 1) * std1**2 ) / (n0 + n1 - 2)) ** 0.5
        cohen_d = (mean1 - mean0) / pooled_std if pooled_std > 0 else 0.0
    else:
        cohen_d = None

    cohen_d_results.append((feat, cohen_d))


25/04/24 09:51:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 09:52:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 09:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
ERROR:root:KeyboardInterrupt while sending command.=======>     (90 + 10) / 100]
Traceback (most recent call last):
  File "/Volumes/CrucialX6/Home/neuro-venv/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Volumes/CrucialX6/Home/neuro-venv/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/homebrew/Cellar/python@3.9/3.9.22/Frameworks/Python.framework/Versions/3.9/lib/python3.9/socket.py", line 716, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [142]:
len(feature_cols)

613

In [143]:
from pyspark.sql.functions import avg, stddev, count
from pyspark.sql import functions as F

# One big grouped DataFrame with all features aggregated
agg_df = (
    train_df
    .select("label", *feature_cols)
    .dropna()
    .groupBy("label")
    .agg(
        *[avg(f).alias(f"{f}_mean") for f in feature_cols],
        *[stddev(f).alias(f"{f}_std") for f in feature_cols],
        *[count(f).alias(f"{f}_n") for f in feature_cols]
    )
)


In [144]:
agg_dict = {row["label"]: row.asDict() for row in agg_df.collect()}


25/04/24 11:02:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 11:02:47 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
25/04/24 11:02:51 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
                                                                                

In [145]:
cohen_d_results = []

for feat in feature_cols:
    try:
        mean0 = agg_dict[0][f"{feat}_mean"]
        mean1 = agg_dict[1][f"{feat}_mean"]
        std0 = agg_dict[0][f"{feat}_std"]
        std1 = agg_dict[1][f"{feat}_std"]
        n0 = agg_dict[0][f"{feat}_n"]
        n1 = agg_dict[1][f"{feat}_n"]

        pooled_std = (( (n0 - 1)*std0**2 + (n1 - 1)*std1**2 ) / (n0 + n1 - 2)) ** 0.5
        cohen_d = (mean1 - mean0) / pooled_std if pooled_std > 0 else 0.0
    except Exception as e:
        print(f"[WARN] Skipping {feat}: {e}")
        cohen_d = None

    cohen_d_results.append((feat, cohen_d))


In [146]:
import os
os.system('say "Cohen-test is done!"')

0

In [147]:
len(cohen_d_results)

613

In [148]:
import pandas as pd

cohen_d_results = pd.DataFrame(cohen_d_results, columns=["Feature", "Cohen_d"])
cohen_d_results = cohen_d_results.sort_values("Cohen_d", ascending=False)


In [149]:
# cohen_d_results.to_csv("cohen_results.csv")

In [150]:
cohen_d_results["Abs_Cohen_d"] = cohen_d_results["Cohen_d"].abs()

In [151]:
cohen_d_results = cohen_d_results.sort_values(by="Abs_Cohen_d", ascending=False)

In [152]:
cohen_d_results.count()

Feature        613
Cohen_d        613
Abs_Cohen_d    613
dtype: int64

In [153]:
cohen_d_results.to_pickle("cohen_results_FILTERD.pkl")
cohen_d_results.to_csv("cohen_results_FILTERD.csv")

# Dimensionality reduction with t-test

In [154]:
# import importlib
# try:
#     importlib.reload(dimensionality_reduction)
# except:
#     pass
# from dimensionality_reduction import apply_pca_model

# train_df = apply_pca_model(train_norm_df, pca_input_cols, pca_model, k_val)
# test_df = apply_pca_model(test_norm_df, pca_input_cols, pca_model, k_val)


In [155]:
# results_df = pd.read_pickle("ttest+cohen_results.pkl")

In [304]:
cohen_min = 0.6
features_of_interest = cohen_d_results[cohen_d_results["Abs_Cohen_d"] > cohen_min]["Feature"].tolist()

In [305]:
print(f"After cohen test we have {len( features_of_interest)} features of interest with a cohen value greater then {cohen_min}")
print(" == features == ")
print(features_of_interest)

After cohen test we have 21 features of interest with a cohen value greater then 0.6
 == features == 
['O2_Alpha_Power', 'O2_custom1_Power', 'T5_Alpha_Power', 'O1_Alpha_Power', 'T5_custom1_Power', 'O2_custom1_HjorthComplexity', 'O2_HjorthComplexity', 'O2_Theta_HjorthComplexity', 'O2_Beta_HjorthComplexity', 'O2_Alpha_HjorthComplexity', 'O2_Delta_HjorthComplexity', 'O1_custom1_Power', 'T5_Beta_HjorthComplexity', 'T5_custom1_HjorthComplexity', 'T5_HjorthComplexity', 'T5_Delta_HjorthComplexity', 'T5_Alpha_HjorthComplexity', 'T5_Theta_HjorthComplexity', 'T6_custom1_Power', 'T6_Alpha_Power', 'O2_Delta_Power']


In [306]:
# Required columns to retain
meta_cols = ["label", "SubjectID", "EpochID"]  # add/remove as needed
selected_cols = meta_cols + features_of_interest
features_of_interest


['O2_Alpha_Power',
 'O2_custom1_Power',
 'T5_Alpha_Power',
 'O1_Alpha_Power',
 'T5_custom1_Power',
 'O2_custom1_HjorthComplexity',
 'O2_HjorthComplexity',
 'O2_Theta_HjorthComplexity',
 'O2_Beta_HjorthComplexity',
 'O2_Alpha_HjorthComplexity',
 'O2_Delta_HjorthComplexity',
 'O1_custom1_Power',
 'T5_Beta_HjorthComplexity',
 'T5_custom1_HjorthComplexity',
 'T5_HjorthComplexity',
 'T5_Delta_HjorthComplexity',
 'T5_Alpha_HjorthComplexity',
 'T5_Theta_HjorthComplexity',
 'T6_custom1_Power',
 'T6_Alpha_Power',
 'O2_Delta_Power']

In [307]:
train_df.head(1)

25/04/24 17:23:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

[Row(label=1, SubjectID='sub-003', EpochID='ep-103', O2_Alpha_Power=-0.8180643949340618, O2_custom1_Power=-0.9545756073044258, T5_Alpha_Power=-0.9151107949358942, O1_Alpha_Power=-0.8890042840019551, T5_custom1_Power=-0.9563129295612013, O2_custom1_HjorthComplexity=-0.5589174693237633, O2_HjorthComplexity=-0.5589174693237633, O2_Theta_HjorthComplexity=-0.5589174693237633, O2_Beta_HjorthComplexity=-0.5589174693237633, O2_Alpha_HjorthComplexity=-0.5589174693237633, O2_Delta_HjorthComplexity=-0.5589174693237633)]

In [308]:
train_df = train_df.select(*selected_cols)
test_df = test_df.select(*selected_cols)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `O1_custom1_Power` cannot be resolved. Did you mean one of the following? [`O2_custom1_Power`, `T5_custom1_Power`, `O1_Alpha_Power`, `O2_Alpha_Power`, `T5_Alpha_Power`].;
'Project [label#11393, SubjectID#10158, EpochID#10159, O2_Alpha_Power#21516705, O2_custom1_Power#21516735, T5_Alpha_Power#21516765, O1_Alpha_Power#21516795, T5_custom1_Power#21516825, O2_custom1_HjorthComplexity#21516855, O2_HjorthComplexity#21516885, O2_Theta_HjorthComplexity#21516915, O2_Beta_HjorthComplexity#21516945, O2_Alpha_HjorthComplexity#21516975, O2_Delta_HjorthComplexity#21517005, 'O1_custom1_Power, 'T5_Beta_HjorthComplexity, 'T5_custom1_HjorthComplexity, 'T5_HjorthComplexity, 'T5_Delta_HjorthComplexity, 'T5_Alpha_HjorthComplexity, 'T5_Theta_HjorthComplexity, 'T6_custom1_Power, 'T6_Alpha_Power, 'O2_Delta_Power]
+- Project [label#11393, SubjectID#10158, EpochID#10159, O2_Alpha_Power#21516705, O2_custom1_Power#2151

In [ ]:
train_df.head(1)

In [ ]:
len(train_df.head(1)[0])

In [ ]:
# Min maxing accross everything

In [ ]:
from pyspark.sql.functions import min as spark_min, max as spark_max

# Example list of selected features (from Cohen's d filtering)
# You already have: features_of_interest
stat_exprs = []

for feature in features_of_interest:
    stat_exprs.append(spark_min(feature).alias(f"{feature}_min"))
    stat_exprs.append(spark_max(feature).alias(f"{feature}_max"))

# Compute min and max for all selected features
feature_ranges = train_df.select(*features_of_interest).agg(*stat_exprs)



# we need to try z-score, z-score by subjet, min_max, min_max by subject

In [ ]:
from pyspark.sql.functions import col, lit

feature_ranges_row = feature_ranges.collect()[0].asDict()

for feat in features_of_interest:
    min_val = feature_ranges_row[f"{feat}_min"]
    max_val = feature_ranges_row[f"{feat}_max"]
    denom = max_val - min_val if max_val != min_val else 1.0  # avoid divide-by-zero

    # Overwrite the original column
    train_df = train_df.withColumn(
        feat,
        (2 * (col(feat) - lit(min_val)) / lit(denom)) - 1
    )

    test_df = test_df.withColumn(
        feat,
        (2 * (col(feat) - lit(min_val)) / lit(denom)) - 1
    )


In [ ]:
train_df.head()

In [ ]:
from pyspark.sql.functions import min as spark_min, max as spark_max

# Example list of selected features (from Cohen's d filtering)
# You already have: features_of_interest
stat_exprs = []

for feature in features_of_interest:
    stat_exprs.append(spark_min(feature).alias(f"{feature}_min"))
    stat_exprs.append(spark_max(feature).alias(f"{feature}_max"))

# Compute min and max for all selected features
feature_ranges = train_df.select(*features_of_interest).agg(*stat_exprs)

In [ ]:
feature_ranges.head()

In [ ]:
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
    
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide

# train_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_df = normalize_by_column_per_subject_wide(test_df, feature_cols)


In [ ]:
print("got here")

# ML time

In [ ]:
train_df.columns

In [ ]:
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()
# spark.stop()

In [ ]:
test_pd.columns.tolist()

In [ ]:
train_pd.columns.tolist()

In [ ]:
import numpy as np

# Convert Spark DenseVectors to regular 2D numpy arrays
# X_train = np.array(train_pd["features"].tolist())
# y_train = train_pd["label"].values

# X_test = np.array(test_pd["features"].tolist())
# y_test = test_pd["label"].values

exclude_cols = ["label", "SubjectID", "EpochID"]
feature_cols = [col for col in train_pd.columns if col not in exclude_cols]

# Features matrix
X_train = train_pd[feature_cols].values
X_test = test_pd[feature_cols].values

# Labels
y_train = train_pd["label"].values
y_test = test_pd["label"].values


In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


In [ ]:
y_train

In [ ]:
X_train[0]

In [ ]:
len(X_train[0])

In [ ]:
# How can we do standard scaler per subject ! !!!  ! ! ! !  !  !! ! !  ! 

In [ ]:
print("work")

In [ ]:
from sklearn.preprocessing import StandardScaler


X_train_scaled = X_train

X_test_scaled = X_test

# making sure min-maxed ! also might change results a little 

# scaler = StandardScaler()

# X_train_scaled = scaler.fit_transform(X_train)

# X_test_scaled = scaler.transform(X_test)

In [ ]:
os.system('say"Ready for ML!"')

In [309]:
%%time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np
import time

# # Step 1: Scale once
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# Step 2: Define hyperparameter grid
k_values = [2, 3, 5, 7, 9, 11]
# k_values = [7, 9, 11, 13, 15, 17, 20]
weights_list = ['uniform', 'distance']
metrics = ['euclidean', 'manhattan']
p_values = [1, 2]  # Only used if metric is 'minkowski'

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]
print("start")

# Step 4: Manual hyperparameter search
for k in k_values:
    for weight in weights_list:
        for metric in metrics:
            for p in p_values:

                if metric != 'minkowski' and p != 2:
                    continue  # p is irrelevant unless using 'minkowski'

                label = f"KNN k={k}, weight={weight}, metric={metric}, p={p}"
                model = KNeighborsClassifier(
                    n_neighbors=k,
                    weights=weight,
                    metric=metric if metric != 'minkowski' else 'minkowski',
                    p=p
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=15, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                    # Step 7: Evaluate on the held-out test set
                    #!! shouldn't we be testing oin the model that is trained on all the folds 1 by 1 (so multiple epochs) ?
                    model.fit(X_train_scaled, y_train)
                    y_test_pred = model.predict(X_test_scaled)
                    test_acc = accuracy_score(y_test, y_test_pred)
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_test, y_test_pred, target_names=target_names))

                    results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))
                    break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 8: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Test: {test_acc:.4f}")


start

=== Cross-Validation: KNN k=2, weight=uniform, metric=euclidean, p=2 ===
Mean Accuracy: 0.6089
Std Deviation: 0.0090
All Fold Scores: [0.6134 0.6036 0.6055 0.6193 0.5993 0.6086 0.6161 0.5994 0.6243 0.5985
 0.6121 0.6151 0.6028 0.6217 0.5943]
Test Accuracy: 0.6216
              precision    recall  f1-score   support

     Control       0.62      0.78      0.69      5524
 Alzheimer's       0.62      0.44      0.51      4624

    accuracy                           0.62     10148
   macro avg       0.62      0.61      0.60     10148
weighted avg       0.62      0.62      0.61     10148

⏱️ Duration: 3.8s

=== Cross-Validation: KNN k=2, weight=uniform, metric=manhattan, p=2 ===
Mean Accuracy: 0.6053
Std Deviation: 0.0089
All Fold Scores: [0.6114 0.6035 0.5971 0.6149 0.5977 0.6078 0.6125 0.591  0.6221 0.5983
 0.6078 0.6087 0.5985 0.6159 0.5927]
Test Accuracy: 0.6166
              precision    recall  f1-score   support

     Control       0.62      0.77      0.69      5524
 Alzheimer

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

# Define models
models = {
    # "KNN-tuned": KNeighborsClassifier(
    #     n_neighbors=3,
    #     weights='distance',
    #     metric='euclidean',
    #     p=1
    # ),
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=8,
        max_features=None,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42
    ),
    # "BaggedSVM": make_pipeline(
    #     BaggingClassifier(
    #         estimator=SVC(kernel='linear', probability=False),
    #         n_estimators=10,
    #         max_samples=0.1,
    #         n_jobs=3,
    #         bootstrap=False,
    #         random_state=42
    #     )
    # ),
    # "SVM": make_pipeline(
    #     SVC(kernel='linear', probability=True)
    # )
}

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            val_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Validation Accuracy: {val_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break

    # Final evaluation on the held-out test set
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    print(f"\n=== Test Set Evaluation: {name} ===")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(classification_report(y_test, y_test_pred, target_names=["Control", "Alzheimer's"]))


In [ ]:
%%time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once (IMPORTANT: transform X_test with same scaler)
# scaler = MinMaxScaler(feature_range=(-1, 1))
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# Step 2: Hyperparameters
layer_configs = [(256, 128, 64), (128, 64, 16)]
activations = ['relu']
alphas = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 5e-1, 1.0]
early_stopping_options = [True] #, False] # !! REMOVED FALSE, TOOK TOO LONG
max_iter = 30000

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force hyperparameter loop
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}, max_iter={max_iter}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=max_iter,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Evaluate on best validation fold
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        break

                # Step 7: Evaluate on held-out test set
                model.fit(X_train_scaled, y_train)
                y_test_pred = model.predict(X_test_scaled)
                test_acc = accuracy_score(y_test, y_test_pred)
                print(f"\n=== Final Test Set Evaluation: {label} ===")
                print(f"Test Accuracy: {test_acc:.4f}")
                print(classification_report(y_test, y_test_pred, target_names=target_names))

                results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 8: Print final summary
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Define the hyperparameter grid
# n_estimators_list = [100, 200, 300]
# learning_rates = [0.1, 0.05]  # try lower learning rate only with more trees
# max_depths = [5, 7, 9]
# subsample_rates = [0.8, 1.0]
n_estimators_list = [100, 200, 300] #, 500]
learning_rates = [0.1, 0.05, 0.01]
learning_rates.sort(reverse=True)  # Smaller learning rate with higher trees
max_depths = [3, 5]#  12]
subsample_rates = [0.6, 0.8, 1.0]   # Add stronger stochasticity

# min_samples_splits = [2, 5, 10]     # Controls node splitting (regularization)
# min_samples_leafs = [1, 3, 5]       # Prevent overfitting small leaves
# max_features_options = ['sqrt', 'log2', None]  # Feature selection per split




# Result tracker
results = []
target_names = ["Control", "Alzheimer's"]

# Brute-force sweep
for n_estimators in n_estimators_list:
    for learning_rate in learning_rates:
        if learning_rate == 0.05 and n_estimators < 200:
            continue  # skip inefficient combos

        for max_depth in max_depths:
            for subsample in subsample_rates:

                label = (f"GBT n_estimators={n_estimators}, lr={learning_rate}, "
                         f"depth={max_depth}, subsample={subsample}")
                model = GradientBoostingClassifier(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    subsample=subsample,
                    random_state=42
                )

                print(f"\n🌲 Cross-Validation: {label}")
                start = time.time()

                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}, Std Dev: {std_acc:.4f}")
                print(f"Fold Scores: {np.round(scores, 4)}")

                # Best fold eval
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
                        y_tr, y_val = y_train[train_idx], y_train[val_idx]

                        model.fit(X_tr, y_tr)
                        y_val_pred = model.predict(X_val)
                        y_tr_pred = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_tr_pred)
                        val_acc = accuracy_score(y_val, y_val_pred)

                        print(f"\n✅ Best Fold Summary: {label}")
                        print(f"Train Accuracy:      {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_val, y_val_pred, target_names=target_names))
                        break

                # Test set eval
                model.fit(X_train_scaled, y_train)
                y_test_pred = model.predict(X_test_scaled)
                test_acc = accuracy_score(y_test, y_test_pred)

                print(f"\n🧪 Test Set Evaluation: {label}")
                print(f"Test Accuracy: {test_acc:.4f}")
                print(classification_report(y_test, y_test_pred, target_names=target_names))

                results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))
                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Sort and show top configs
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== 🏆 Top Gradient Boosting Models by CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
%%time
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: Scale data for SVMs
scaler = StandardScaler()
X_train_svm = scaler.fit_transform(X_train)
X_test_svm = scaler.transform(X_test)

# Step 2: Define hyperparameter grids
C_values = [0.01, 0.1, 1, 10]
n_estimators_list = [5, 10, 20]
max_samples_list = [0.1, 0.5, 1.0]
bootstrap_options = [False, True]

# Step 3: Track results
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Hyperparameter tuning
for C in C_values:
    for n_est in n_estimators_list:
        for max_samp in max_samples_list:
            for bootstrap in bootstrap_options:
                
                label = f"BaggedSVM C={C}, est={n_est}, max_samples={max_samp}, bootstrap={bootstrap}"
                model = make_pipeline(
                    BaggingClassifier(
                        estimator=SVC(C=C, kernel='linear', probability=False),
                        n_estimators=n_est,
                        max_samples=max_samp,
                        bootstrap=bootstrap,
                        n_jobs=3,
                        random_state=42
                    )
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                scores = cross_val_score(model, X_train_svm, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Best fold deep dive
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, val_idx) in enumerate(skf.split(X_train_svm, y_train)):
                    if i == best_fold_index:
                        X_tr, X_val = X_train_svm[train_idx], X_train_svm[val_idx]
                        y_tr, y_val = y_train[train_idx], y_train[val_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_val = model.predict(X_val)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_val, y_pred_val)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_val, y_pred_val, target_names=target_names))

                        break

                # Final test set evaluation
                model.fit(X_train_svm, y_train)
                y_test_pred = model.predict(X_test_svm)
                test_acc = accuracy_score(y_test, y_test_pred)

                print(f"\n=== Test Set Evaluation: {label} ===")
                print(f"Test Accuracy: {test_acc:.4f}")
                print(classification_report(y_test, y_test_pred, target_names=target_names))

                results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 5: Print top models
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<90} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
%%time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: No need to scale for trees
X_train_tree = X_train
X_test_tree = X_test

# Step 2: More complex hyperparameter grid
max_depths = [10, 15, 20, None]  # None = fully grow the tree
min_samples_splits = [2, 3, 5]   # Smaller split thresholds
min_samples_leafs = [1, 2]       # Smaller leaves allowed

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Manual hyperparameter search
for depth in max_depths:
    for min_split in min_samples_splits:
        for min_leaf in min_samples_leafs:

            label = f"DecisionTree max_depth={depth}, min_split={min_split}, min_leaf={min_leaf}"
            model = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_split=min_split,
                min_samples_leaf=min_leaf,
                max_features=None,  # use all features
                ccp_alpha=0.0,      # disable pruning
                random_state=42
            )

            print(f"\n=== Cross-Validation: {label} ===")
            start = time.time()

            # Step 5: Cross-validation scores
            scores = cross_val_score(model, X_train_tree, y_train, cv=5, scoring='accuracy', n_jobs=3)
            mean_acc, std_acc = scores.mean(), scores.std()
            print(f"Mean Accuracy: {mean_acc:.4f}")
            print(f"Std Deviation: {std_acc:.4f}")
            print(f"All Fold Scores: {np.round(scores, 4)}")

            # Step 6: Best fold evaluation
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            best_fold_index = np.argmax(scores)
            for i, (train_idx, test_idx) in enumerate(skf.split(X_train_tree, y_train)):
                if i == best_fold_index:
                    X_tr, X_te = X_train_tree[train_idx], X_train_tree[test_idx]
                    y_tr, y_te = y_train[train_idx], y_train[test_idx]

                    model.fit(X_tr, y_tr)
                    y_pred_test = model.predict(X_te)
                    y_pred_train = model.predict(X_tr)

                    train_acc = accuracy_score(y_tr, y_pred_train)
                    test_acc = accuracy_score(y_te, y_pred_test)

                    print(f"\n=== Best Fold Summary: {label} ===")
                    print(f"Train Accuracy: {train_acc:.4f}")
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_te, y_pred_test, target_names=target_names))

                    results.append((label, mean_acc, std_acc, train_acc, test_acc))
                    break

            print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np


# Define models
models = { # NEED TO GO THORUGH AND MAKE SURE DEFAULTS ARE OK, SHOULD NOT HAVE HAD STANDARD SCALER FOR SVM'S
    # "KNN-tuned": KNeighborsClassifier(
    # n_neighbors=3,
    # weights='distance',
    # metric='euclidean',
    # p=1
    # ),
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=8,
        max_features=None,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42
    ),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
        # Faster ensemble of SVMs
    "BaggedSVM": make_pipeline(
        # StandardScaler(),
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),

    # Classic SVM (can be slow on big data)
    "SVM": make_pipeline(
        # StandardScaler(), 
        SVC(kernel='linear',
        probability=True)
    )
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break


# ML experiments and tuning

Notes and concerns:
 Are we folding correclty, it looks like we are retraining the best fold, ??? super weird
 Also seems like we need to do more iterations for SVM ... we can get some more out of it 

 Since hte neural nets that havve best aucuracy almost have 1000% in training, should try and do regurlization to try and 'even' out the accuracy between the two

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
import numpy as np
import time


import warnings
from sklearn.exceptions import ConvergenceWarning

# Suppress ConvergenceWarnings only
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Set up test space
kernels = ['linear', 'rbf', 'sigmoid']
Cs = [0.1, 1, 10]
gammas = ['scale', 0.01]  # for rbf/sigmoid
n_estimators = 10
max_iter = 10000

# Replace with your target class names
target_names = ["Control", "Alzheimer's"]

# Loop over kernel/C/gamma
for kernel in kernels:
    for C in Cs:
        if kernel == 'linear':
            label = f"BaggedSVM - linear, C={C}"
            base_model = SVC(kernel=kernel, C=C, probability=False, max_iter=max_iter)
            model = make_pipeline(
                BaggingClassifier(
                    estimator=base_model,
                    n_estimators=n_estimators,
                    max_samples=0.1,
                    n_jobs=3,
                    bootstrap=False,
                    random_state=42
                )
            )
            print(f"\n=== Cross-Validation: {label} ===")
            start = time.time()

            scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
            print(f"Mean Accuracy: {scores.mean():.4f}")
            print(f"Standard Deviation: {scores.std():.4f}")
            print(f"All Fold Scores: {np.round(scores, 4)}")

            best_fold_index = np.argmax(scores)
            skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
            for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
                if i == best_fold_index:
                    X_tr, X_te = X_train[train_index], X_train[test_index]
                    y_tr, y_te = y_train[train_index], y_train[test_index]

                    model.fit(X_tr, y_tr)
                    y_pred_test = model.predict(X_te)
                    y_pred_train = model.predict(X_tr)

                    test_acc = accuracy_score(y_te, y_pred_test)
                    train_acc = accuracy_score(y_tr, y_pred_train)

                    print(f"\n=== Best Fold Summary: {label} ===")
                    print(f"Train Accuracy: {train_acc:.4f}")
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_te, y_pred_test, target_names=target_names))
                    break

            print(f"⏱️ Duration: {time.time() - start:.1f} sec")

        elif kernel in ['rbf', 'sigmoid']:
            for gamma in gammas:
                label = f"BaggedSVM - {kernel}, C={C}, gamma={gamma}"
                base_model = SVC(kernel=kernel, C=C, gamma=gamma, probability=False, max_iter=max_iter)
                model = make_pipeline(
                    BaggingClassifier(
                        estimator=base_model,
                        n_estimators=n_estimators,
                        max_samples=0.1,
                        n_jobs=3,
                        bootstrap=False,
                        random_state=42
                    )
                )
                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
                print(f"Mean Accuracy: {scores.mean():.4f}")
                print(f"Standard Deviation: {scores.std():.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                best_fold_index = np.argmax(scores)
                skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
                for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train[train_index], X_train[test_index]
                        y_tr, y_te = y_train[train_index], y_train[test_index]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        test_acc = accuracy_score(y_te, y_pred_test)
                        train_acc = accuracy_score(y_tr, y_pred_train)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f} sec")


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once
scaler = MinMaxScaler(feature_range=(-1, 1))  # You can use (0, 1) if you prefer
X_train_scaled = scaler.fit_transform(X_train)

# Step 2: Define hyperparameter grids
layer_configs = [(100,), (128,), (128, 64), (256, 128, 64)]
activations = ['relu', 'tanh']
alphas = [1e-4, 1e-3, 1e-2]
early_stopping_options = [True, False]

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force loop over hyperparameter combinations
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=10000,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation accuracy scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Find best fold for detailed summary
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
%%time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: Scale features if needed (optional for GB, but useful with continuous variables)
X_train_boost = X_train_scaled  # assuming you already have this
X_test_boost = X_test_scaled

# Step 2: Define hyperparameter grid for more complex boosting
n_estimators_list = [100, 200]
learning_rates = [0.05, 0.1]
max_depths = [3, 5, 7]
min_samples_leafs = [1, 3]

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Manual hyperparameter search
for n_est in n_estimators_list:
    for lr in learning_rates:
        for depth in max_depths:
            for min_leaf in min_samples_leafs:

                label = f"GradBoost n={n_est}, lr={lr}, depth={depth}, min_leaf={min_leaf}"
                model = GradientBoostingClassifier(
                    n_estimators=n_est,
                    learning_rate=lr,
                    max_depth=depth,
                    min_samples_leaf=min_leaf,
                    random_state=42
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_boost, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_boost, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_boost[train_idx], X_train_boost[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

import os

# Define models
models = { # NEED TO GO THORUGH AND MAKE SURE DEFAULTS ARE OK, SHOULD NOT HAVE HAD STANDARD SCALER FOR SVM'S
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    # "DecisionTree": DecisionTreeClassifier(
    #     max_depth=8,
    #     max_features=None,
    #     min_samples_leaf=10,
    #     min_samples_split=5,
    #     random_state=42
    # ),
    # "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
        # Faster ensemble of SVMs
    # "BaggedSVM": make_pipeline(
    #     StandardScaler(),
    #     BaggingClassifier(
    #         estimator=SVC(kernel='linear', probability=False),
    #         n_estimators=10,
    #         max_samples=0.1,
    #         n_jobs=3,
    #         bootstrap=False,
    #         random_state=42
    #     )
    # ),
    # Classic SVM (can be slow on big data)
    "BaggedSVM": make_pipeline(
        # StandardScaler(),
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break

os.system('say "SVM is done!"')

# ML Testing/tuning # best for nets, tanh 200 hidden layers

In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

# Define models
models = {
    # "KNN": KNeighborsClassifier(),
    # "SVM": make_pipeline(StandardScaler(), SVC(probability=True)),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(200,), activation='tanh', max_iter=10000, random_state=42)
    # "DecisionTree": DecisionTreeClassifier(
    #     max_depth=8,
    #     max_features=None,
    #     min_samples_leaf=10,
    #     min_samples_split=5,
    #     random_state=42
    # ),
    # "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break


In [ ]:
model = DecisionTreeClassifier(
    max_depth=8,
    max_features=None,
    min_samples_leaf=10,
    min_samples_split=5,
    random_state=42
)

print(f"\n=== Cross-Validation: Regularized DecisionTree ===")

# Cross-validation scores
scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)

print(f"Mean Accuracy: {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")
print(f"All Fold Scores: {np.round(scores, 4)}")

# Find the best-performing fold
best_fold_index = np.argmax(scores)

# Evaluate best fold manually
skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
    if i == best_fold_index:
        X_tr, X_te = X_train[train_index], X_train[test_index]
        y_tr, y_te = y_train[train_index], y_train[test_index]

        model.fit(X_tr, y_tr)

        y_pred_train = model.predict(X_tr)
        y_pred_test = model.predict(X_te)

        train_acc = accuracy_score(y_tr, y_pred_train)
        test_acc = accuracy_score(y_te, y_pred_test)

        print(f"\n=== Best Fold Summary: Regularized DecisionTree ===")
        print(f"Train Accuracy: {train_acc:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}")
        print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
        break


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    'max_depth': [4, 6, 8],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 'log2', None]
}

tree = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(tree, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)